In [2]:
import heapq
from collections import defaultdict

def time_to_minutes(t):
    return int(t[:2]) * 60 + int(t[2:])

def minutes_to_time(m):
    m = m % (24*60)
    return f"{m//60:02d}{m%60:02d}"

def read_flights(filename):
    flights = defaultdict(list)
    with open(filename) as f:
        for line in f:
            if not line.strip():
                continue
            parts = line.strip().split()
            origin, dep, dest, arr = parts
            dep_min = time_to_minutes(dep)
            arr_min = time_to_minutes(arr)
            if arr_min < dep_min:
                arr_min += 24*60
            flights[origin].append((dep_min, dest, arr_min))
    return flights

def dijkstra(flights, source, start_time):
    layover = 40
    heap = []
    heapq.heappush(heap, (time_to_minutes(start_time), source, []))
    best_arrival = dict()
    results = dict()
    while heap:
        curr_time, airport, path = heapq.heappop(heap)
        if airport in best_arrival and curr_time >= best_arrival[airport]:
            continue
        best_arrival[airport] = curr_time
        results[airport] = (curr_time, path)
        for dep_min, dest, arr_min in flights[airport]:
            earliest_dep = curr_time if not path else curr_time + layover
            if dep_min >= earliest_dep:
                heapq.heappush(heap, (arr_min, dest, path + [(airport, dep_min, dest, arr_min)]))
    return results

def print_results(results, source, start_time):
    start_min = time_to_minutes(start_time)
    for airport in sorted(results):
        if airport == source:
            continue
        arr_time, path = results[airport]
        print(f"\nDestination: {airport}")
        print(f"Total travel time: {arr_time - start_min} minutes")
        prev_arr = start_min
        for step in path:
            o, dpt, d, arr = step
            print(f"  {o} -> {d} | Depart: {minutes_to_time(dpt)} Arrive: {minutes_to_time(arr)}")
            if o != source:
                layover = dpt - prev_arr
                print(f"    Layover at {o}: {layover} minutes")
            prev_arr = arr
        print(f"Final arrival at {airport}: {minutes_to_time(arr_time)}")

if __name__ == "__main__":
    flights = read_flights("input.txt")
    source = "TEZ"
    start_time = "0600"
    results = dijkstra(flights, source, start_time)
    print_results(results, source, start_time)
